# Extraction KYC – Documents scannés et modèle Qwen local
Ce notebook réalise l'extraction sélective des PDF depuis un ZIP, vérifie l'existence des fichiers par client, identifie les clients cibles disposant de `JUSTIFICATIF IDENTITE.PDF`, puis extrait les informations structurées à l'aide d'OCR robuste et du modèle Qwen chargé localement.

## 1. Installation des dépendances

In [ ]:
# Cellule 1 : Installation des dépendances
# Adapter les versions en fonction de votre CUDA / environnement Domino.

# Bibliothèques Python
!pip install pandas==2.2.3 openpyxl==3.1.5 tqdm==4.66.5
!pip install pdf2image==1.17.0 pytesseract==0.3.13 opencv-python-headless==4.10.0.84 pillow==10.4.0 numpy==1.26.4
!pip install transformers==4.51.3 tokenizers==0.21.0 accelerate==0.34.2
!pip install vllm==0.8.5.post1

# Dépendances système (si exécuté dans un environnement Linux type Domino / JupyterLab)
# Installe poppler-utils pour pdf2image et tesseract + langues
!apt-get update -qq && apt-get install -y -qq \
    poppler-utils \
    tesseract-ocr \
    tesseract-ocr-fra \
    tesseract-ocr-ara \
    tesseract-ocr-spa \
    tesseract-ocr-deu \
    tesseract-ocr-ita \
    tesseract-ocr-por \
    tesseract-ocr-nld \
    tesseract-ocr-rus \
    tesseract-ocr-chi-sim

print("Installations terminées.")

## 2. Imports et configuration

In [ ]:
# Cellule 2 : Imports et configuration générale
import os
import sys
import json
import re
import shutil
import zipfile
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from pdf2image import convert_from_path
from PIL import Image
import cv2
import pytesseract
from pytesseract import TesseractError
import torch

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
# Paramètres généraux
# ------------------------------------------------------------------
ZIP_PATH = "/mnt/data/kyc_documents.zip"          # À ADAPTER
OUTPUT_DIR = Path("/mnt/data/extracted_kyc")      # Dossiers clients extraits
REPORT_DIR = Path("/mnt/data/reports")            # Rapports d'existence
OCR_DIR = Path("/mnt/data/ocr_texts")             # Textes OCR sauvegardés
RESULTS_DIR = Path("/mnt/data/results")           # Résultats JSON/Excel

for d in [OUTPUT_DIR, REPORT_DIR, OCR_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Fichiers à extraire uniquement
REQUIRED_FILES = [
    "JUSTIFICATIF IDENTITE.PDF",
    "JUSTIFICATIF DOMICILE.PDF",
    "CONVENTION COMPTE.PDF",
    "FATCA.PDF",
    "CARTON SIGNATUTE.PDF"   # typo probable de "CARTON SIGNATURE.PDF"
]

TARGET_FILE = "JUSTIFICATIF IDENTITE.PDF"

# Alias : pour gérer la faute de frappe éventuelle
# On normalise les noms pour comparer sans accents / majuscules
def normalize_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    return s.upper().strip()

ALIASES = {
    normalize_name("CARTON SIGNATURE.PDF"): normalize_name("CARTON SIGNATUTE.PDF")
}

REQUIRED_NORM = {normalize_name(f) for f in REQUIRED_FILES}

print("Configuration terminée.")

## 3. Extraction sélective depuis le ZIP et rapport d’existence

In [ ]:
# Cellule 3 : Extraction des PDF requis et création du rapport d'existence

def sanitize_client_id(name: str) -> str:
    """Nettoie l'identifiant client pour éviter les caractères problématiques."""
    return re.sub(r'[\\/*?:"<>|]', "_", name)

def extract_required_files(zip_path, output_dir, required_files, alias_map=None):
    """
    Extrait uniquement les fichiers requis du zip.
    Retourne un DataFrame avec l'existence de chaque fichier par client.
    """
    alias_map = alias_map or {}
    required_norm = {normalize_name(f) for f in required_files}
    
    # D'abord, identifier tous les clients présents dans le zip
    client_ids = set()
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            parts = Path(info.filename).parts
            if len(parts) >= 2:
                # Si le premier niveau est le nom du zip, on prend le suivant
                if parts[0].lower() == Path(zip_path).stem.lower():
                    cid = sanitize_client_id(parts[1]) if len(parts) > 1 else "RACINE"
                else:
                    cid = sanitize_client_id(parts[0])
            else:
                cid = "RACINE"
            client_ids.add(cid)
    
    # Initialiser la structure d'existence
    existence = {cid: {} for cid in client_ids}
    
    # Extraire les fichiers correspondants
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            parts = Path(info.filename).parts
            basename = Path(info.filename).name
            norm_basename = normalize_name(basename)
            
            # Appliquer les alias éventuels
            canonical = alias_map.get(norm_basename, norm_basename)
            
            if canonical in required_norm:
                if len(parts) >= 2:
                    if parts[0].lower() == Path(zip_path).stem.lower():
                        client_id = sanitize_client_id(parts[1]) if len(parts) > 1 else "RACINE"
                    else:
                        client_id = sanitize_client_id(parts[0])
                else:
                    client_id = "RACINE"
                
                client_dir = Path(output_dir) / client_id
                client_dir.mkdir(parents=True, exist_ok=True)
                
                dest = client_dir / basename
                # Si un fichier du même nom existe déjà, on ajoute un suffixe CRC
                if dest.exists():
                    suffix = f"_{info.CRC}"
                    dest = client_dir / f"{Path(basename).stem}{suffix}{Path(basename).suffix}"
                
                with zf.open(info) as src, open(dest, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                
                # Marquer l'existence pour ce client et ce fichier canonique
                existence[client_id][canonical] = True
    
    # Construire le DataFrame de rapport
    rows = []
    for cid in sorted(existence.keys()):
        row = {"client_id": cid}
        for f in required_files:
            norm_f = normalize_name(f)
            row[f] = existence.get(cid, {}).get(norm_f, False)
        row["target"] = row[TARGET_FILE]
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df

# Lancer l'extraction
report = extract_required_files(ZIP_PATH, OUTPUT_DIR, REQUIRED_FILES, ALIASES)

# Sauvegarder le rapport
report.to_csv(REPORT_DIR / "existence_report.csv", index=False)
print("Rapport d'existence créé :")
print(report.head())

# Liste des clients cibles : ceux qui possèdent JUSTIFICATIF IDENTITE.PDF
targets = report.loc[report[TARGET_FILE], "client_id"].tolist()
pd.DataFrame({"client_id": targets}).to_csv(REPORT_DIR / "target_clients.csv", index=False)

print(f"\nNombre de clients cibles : {len(targets)}")
print("Premiers clients cibles :", targets[:10])

## 4. Fonctions OCR : prétraitement des scans et extraction du texte

In [ ]:
# Cellule 4 : OCR robuste des documents scannés

# Langues OCR à essayer, dans l'ordre. Ajoutez/enlevez selon les packs installés.
OCR_LANGS = ["eng+fra+ara", "eng+fra", "eng"]

def preprocess_image(pil_image):
    """
    Prétraite une image scannée :
    - inversion si le fond est sombre,
    - correction d'orientation via Tesseract OSD,
    - redressement léger,
    - amélioration du contraste et binarisation adaptative.
    """
    img = np.array(pil_image.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 1. Inversion si le scan est inversé (fond majoritairement sombre)
    if np.mean(gray) < 127:
        gray = cv2.bitwise_not(gray)
    
    # 2. Correction d'orientation avec Tesseract OSD
    try:
        osd = pytesseract.image_to_osd(Image.fromarray(gray))
        angle = int(osd.get("rotate", 0))
        if angle == 90:
            gray = cv2.rotate(gray, cv2.ROTATE_90_CLOCKWISE)
        elif angle == 180:
            gray = cv2.rotate(gray, cv2.ROTATE_180)
        elif angle == 270:
            gray = cv2.rotate(gray, cv2.ROTATE_90_COUNTERCLOCKWISE)
    except Exception:
        # OSD peut échouer sur des documents très bruités, on continue
        pass
    
    # 3. Redressement léger (deskew)
    try:
        ys, xs = np.where(gray < 128)
        if len(xs) > 0:
            coords = np.column_stack((xs, ys)).astype(np.float32)
            angle = cv2.minAreaRect(coords)[-1]
            if angle < -45:
                angle = -(90 + angle)
            else:
                angle = -angle
            if abs(angle) > 0.5:
                h, w = gray.shape[:2]
                M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
                gray = cv2.warpAffine(gray, M, (w, h),
                                      flags=cv2.INTER_CUBIC,
                                      borderMode=cv2.BORDER_REPLICATE)
    except Exception:
        pass
    
    # 4. Amélioration du contraste
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    
    # 5. Binarisation adaptative (texte noir sur fond blanc)
    processed = cv2.adaptiveThreshold(gray, 255,
                                      cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY, 31, 11)
    return Image.fromarray(processed)

def ocr_image(pil_image):
    """
    Lance l'OCR sur une image prétraitée avec plusieurs combinaisons de langues
    et de modes de segmentation, pour maximiser les chances de lecture.
    """
    for langs in OCR_LANGS:
        for psm in ["6", "3"]:
            try:
                return pytesseract.image_to_string(
                    pil_image,
                    lang=langs,
                    config=f"--oem 3 --psm {psm}"
                )
            except TesseractError:
                continue
    # Dernier recours
    try:
        return pytesseract.image_to_string(pil_image, lang="eng")
    except Exception:
        return ""

def pdf_to_ocr_text(pdf_path, dpi=300, max_pages=None):
    """
    Convertit toutes les pages d'un PDF scanné en texte OCR.
    """
    pages = convert_from_path(str(pdf_path), dpi=dpi, fmt="jpeg")
    page_texts = []
    
    for i, page in enumerate(pages, 1):
        if max_pages is not None and i > max_pages:
            break
        processed = preprocess_image(page)
        text = ocr_image(processed)
        page_texts.append(f"--- Page {i} ---\n{text}")
    
    return "\n".join(page_texts)

print("Fonctions OCR prêtes.")

## 5. Chargement du modèle Qwen local et fonctions d’extraction LLM

In [ ]:
# Cellule 5 : Chargement du modèle Qwen et parsing JSON

MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"
FP8_KERNEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-kernels-community/finegrained-fp8/v4"

# Optionnel : ajouter le kernel FP8 au PYTHONPATH
sys.path.insert(0, FP8_KERNEL_PATH)

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# Détection du nombre de GPUs
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"GPUs disponibles : {num_gpus}")

# Chargement du modèle
# Si le chargement échoue avec quantization="fp8", essayez quantization=None ou "auto"
try:
    llm = LLM(
        model=MODEL_PATH,
        trust_remote_code=True,
        tensor_parallel_size=num_gpus,
        dtype="auto",
        gpu_memory_utilization=0.90,
        max_model_len=8192,
        quantization="fp8"
    )
except Exception as e:
    print(f"Erreur avec quantization='fp8', tentative avec auto : {e}")
    llm = LLM(
        model=MODEL_PATH,
        trust_remote_code=True,
        tensor_parallel_size=num_gpus,
        dtype="auto",
        gpu_memory_utilization=0.90,
        max_model_len=8192,
        quantization="auto"
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
sampling_params = SamplingParams(temperature=0.0, max_tokens=1024, top_p=0.95)

# ------------------------------------------------------------------
# Prompt d'extraction
# ------------------------------------------------------------------
SYSTEM_PROMPT = (
    "You are an expert in KYC document data extraction. "
    "You parse OCR text from identity documents (passport, national ID, residence permit, etc.). "
    "The OCR may contain errors and be in French, English, Arabic, Spanish, etc. "
    "Return only a valid JSON object."
)

def build_messages(ocr_text: str, max_chars: int = 12000):
    """Construit les messages pour le LLM."""
    truncated = ocr_text[:max_chars]
    user_prompt = f"""
Extract the following fields from the OCR text of an identity document.
Return a JSON object with exactly these keys:
{{
  "document_type": "PASSPORT | NATIONAL_ID | RESIDENCE_PERMIT | DRIVER_LICENSE | OTHER",
  "document_number": "",
  "surname": "",
  "given_names": "",
  "date_of_birth": "",
  "place_of_birth": "",
  "nationality": "",
  "gender": "",
  "issue_date": "",
  "expiry_date": "",
  "issuing_authority": "",
  "mrz_raw": "",
  "raw_address": "",
  "notes": ""
}}

Rules:
- Use null for missing values.
- Normalize dates to YYYY-MM-DD if possible, otherwise keep the original text.
- document_type must be one of the values listed.
- given_names: all first/middle names; surname: last name.
- mrz_raw: exactly the two or three-line MRZ if present, otherwise null.
- If uncertain, use your best judgement.
- Return only JSON, no markdown, no explanation.

OCR text:
{truncated}
"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

def parse_json_output(text: str) -> dict:
    """Extrait et parse le JSON renvoyé par le LLM."""
    text = text.strip()
    # Supprimer les fences markdown éventuelles
    if text.startswith("```"):
        text = re.sub(r"```(?:json)?", "", text).replace("```", "").strip()
    
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        return {"error": "No JSON found", "raw_output": text}
    
    try:
        data = json.loads(text[start:end+1])
        return data
    except json.JSONDecodeError as e:
        return {"error": f"JSON decode error: {e}", "raw_output": text}

print("Modèle Qwen chargé et fonctions prêtes.")

## 6. Extraction des informations pour les clients cibles

In [ ]:
# Cellule 6 : Boucle d'extraction sur les clients cibles

def find_target_file(client_dir: Path, target_file: str) -> Path:
    """Retrouve le fichier cible exact dans le dossier client, insensible à la casse/accents."""
    target_norm = normalize_name(target_file)
    for f in client_dir.iterdir():
        if f.is_file() and normalize_name(f.name) == target_norm:
            return f
    return None

results = []
failed = []

for client_id in tqdm(targets, desc="Extraction KYC"):
    client_dir = OUTPUT_DIR / client_id
    pdf_file = find_target_file(client_dir, TARGET_FILE)
    
    if pdf_file is None:
        failed.append({
            "client_id": client_id,
            "error": "Fichier cible introuvable dans le dossier extrait"
        })
        continue
    
    try:
        # 1. OCR du PDF
        ocr_text = pdf_to_ocr_text(pdf_file)
        
        # 2. Sauvegarde du texte OCR (pour debug / audit)
        ocr_file = OCR_DIR / f"{client_id}.txt"
        ocr_file.write_text(ocr_text, encoding="utf-8")
        
        # 3. Préparation du prompt et génération
        messages = build_messages(ocr_text)
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        outputs = llm.generate([prompt], sampling_params=sampling_params)
        raw_llm_output = outputs[0].outputs[0].text
        
        # 4. Parsing JSON
        parsed = parse_json_output(raw_llm_output)
        parsed["client_id"] = client_id
        parsed["ocr_file"] = str(ocr_file)
        parsed["raw_llm_output"] = raw_llm_output
        
        results.append(parsed)
        
    except Exception as e:
        failed.append({
            "client_id": client_id,
            "error": str(e)
        })

print(f"\nExtraction terminée : {len(results)} succès, {len(failed)} échecs.")

# Sauvegarde intermédiaire en JSON
with open(RESULTS_DIR / "extraction_results_raw.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

## 7. Export final en JSON et Excel

In [ ]:
# Cellule 7 : Exports finaux

# Résultats structurés
results_df = pd.DataFrame(results)

# Export JSON
results_df.to_json(
    RESULTS_DIR / "extraction_results.json",
    orient="records",
    force_ascii=False,
    indent=2
)

# Export Excel
results_df.to_excel(
    RESULTS_DIR / "extraction_results.xlsx",
    index=False,
    engine="openpyxl"
)

# Export des échecs éventuels
if failed:
    pd.DataFrame(failed).to_csv(
        RESULTS_DIR / "failed_extractions.csv",
        index=False
    )

print("Exports terminés :")
print(f" - JSON : {RESULTS_DIR / 'extraction_results.json'}")
print(f" - Excel : {RESULTS_DIR / 'extraction_results.xlsx'}")
if failed:
    print(f" - Échecs : {RESULTS_DIR / 'failed_extractions.csv'}")

# Aperçu du DataFrame final
results_df.head()